In [22]:
# Cell 0 — Project Imports

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F

In [23]:
# Cell 1 — Spacing 변경에 따른 Target Shape 계산

def calculate_target_shape(
    original_shape_dhw: tuple[int, int, int],             # [D, H, W]
    original_spacing_dhw_mm: tuple[float, float, float],  # [D, H, W] mm
    target_spacing_dhw_mm: tuple[float, float, float],    # [D, H, W] mm
) -> tuple[int, int, int]:                                # [D, H, W]
    """Physical size를 유지하는 target voxel-grid Shape 계산."""
    
    # 원본 voxel 개수를 계산용 Tensor로 변환
    original_shape = torch.tensor(
        original_shape_dhw,
        dtype=torch.float64,
    ) # [3] = [D, H, W]
    
    # 원본 voxel 한 칸의 실제 크기
    original_spacing = torch.tensor(
        original_spacing_dhw_mm,
        dtype=torch.float64,
    )  # [3] = [D, H, W] mm
    
    # Resampling 이후 사용할 voxel 크기
    target_spacing = torch.tensor(
        target_spacing_dhw_mm,
        dtype=torch.float64,
    )  # [3] = [D, H, W] mm
    

    # 원본 volume이 덮는 physical size 계산:
    # [16, 32, 20] × [2, 1, 3] mm = [32, 32, 60] mm
    physical_size_dhw_mm = (
        original_shape
        * original_spacing
    )  # [3]
    
    # 동일한 physical size를 target spacing으로 나눈 target voxel 개수 계산
    # [32, 32, 60] mm ÷ [1, 1, 1] = [32, 32, 60]
    target_shape_float = (
        physical_size_dhw_mm
        / target_spacing
    ) # [3]
    
    target_shape_tensor = torch.round(
        target_shape_float
    ).to(
        dtype=torch.int64
    )  # [3]
    
    target_shape_dhw = (
        int(target_shape_tensor[0].item()),
        int(target_shape_tensor[1].item()),
        int(target_shape_tensor[2].item()),
    )

    return target_shape_dhw



# 원본 voxel grid 정의
original_shape_dhw: tuple[int, int, int] = (
    16,
    32,
    20,
)

original_spacing_dhw_mm: tuple[
    float,
    float,
    float,
] = (
    2.0,
    1.0,
    3.0,
)


# Isotropic 1 mm target spacing 정의
#
# Isotropic:
# 모든 spatial axis에서 동일한 voxel spacing
target_spacing_dhw_mm: tuple[
    float,
    float,
    float,
] = (
    1.0,
    1.0,
    1.0,
)


# Physical size를 유지하는 target Shape 계산
target_shape_dhw = calculate_target_shape(
    original_shape_dhw=original_shape_dhw,
    original_spacing_dhw_mm=(
        original_spacing_dhw_mm
    ),
    target_spacing_dhw_mm=(
        target_spacing_dhw_mm
    ),
)


# Resampling 전 physical size 계산
original_physical_size_dhw_mm = (
    torch.tensor(
        original_shape_dhw,
        dtype=torch.float64,
    )
    * torch.tensor(
        original_spacing_dhw_mm,
        dtype=torch.float64,
    )
)  # [3]


# Resampling 후 physical size 계산
resampled_physical_size_dhw_mm = (
    torch.tensor(
        target_shape_dhw,
        dtype=torch.float64,
    )
    * torch.tensor(
        target_spacing_dhw_mm,
        dtype=torch.float64,
    )
)  # [3]


print(
    "Original shape [D, H, W]:",
    original_shape_dhw,
)

print(
    "Original spacing [D, H, W] mm:",
    original_spacing_dhw_mm,
)

print(
    "Target spacing [D, H, W] mm:",
    target_spacing_dhw_mm,
)

print(
    "Target shape [D, H, W]:",
    target_shape_dhw,
)

print(
    "Original physical size [D, H, W] mm:",
    original_physical_size_dhw_mm,
)

print(
    "Resampled physical size [D, H, W] mm:",
    resampled_physical_size_dhw_mm,
)

Original shape [D, H, W]: (16, 32, 20)
Original spacing [D, H, W] mm: (2.0, 1.0, 3.0)
Target spacing [D, H, W] mm: (1.0, 1.0, 1.0)
Target shape [D, H, W]: (32, 32, 60)
Original physical size [D, H, W] mm: tensor([32., 32., 60.], dtype=torch.float64)
Resampled physical size [D, H, W] mm: tensor([32., 32., 60.], dtype=torch.float64)


In [24]:
# Cell 2 — Trilinear Interpolation으로 CT Image Resampling

def resample_ct_image(
    ct_image_bcdhw: torch.Tensor,         # [B, C, D, H, W]
    target_shape_dhw: tuple[int, int, int],
) -> torch.Tensor:                        # [B, C, D_target, H_target, W_target]
    """CT image를 target spatial Shape로 trilinear resampling."""

    # CT image의 spatial grid만 target Shape로 변경
    #
    # Batch B와 Channel C는 그대로 유지
    # D, H, W만 target_shape_dhw로 변경
    resampled_ct_image = F.interpolate(
        ct_image_bcdhw,
        size=target_shape_dhw,
        mode="trilinear",
        align_corners=False,
    )
    
    return resampled_ct_image

# 각 spatial axis의 정규화 좌표 생성
#
# 실제 mm 좌표가 아니라 synthetic volume 생성을 위한
# -1부터 +1 사이의 상대 좌표
depth_coordinates = torch.linspace(
    -1.0,
    1.0,
    steps=original_shape_dhw[0],
)  # [D=16]

height_coordinates = torch.linspace(
    -1.0,
    1.0,
    steps=original_shape_dhw[1],
)  # [H=32]

width_coordinates = torch.linspace(
    -1.0,
    1.0,
    steps=original_shape_dhw[2],
)  # [W=20]


# 1D 좌표 세 개를 3D coordinate grid로 확장
#
# 각 결과 Shape:
# [D=16, H=32, W=20]
grid_d, grid_h, grid_w = torch.meshgrid(
    depth_coordinates,
    height_coordinates,
    width_coordinates,
    indexing="ij",
)


# 각 voxel과 volume 중심 사이의 squared distance 계산
squared_radius_dhw = (
    grid_d.square()
    + grid_h.square()
    + grid_w.square()
)  # [D=16, H=32, W=20]


# 중심은 약 100 HU, 바깥은 약 -1000 HU인
# 부드러운 synthetic CT intensity 생성
#
# exp(-4 × radius²):
# 중심에서 1에 가까움
# 바깥으로 갈수록 0에 가까움
synthetic_ct_dhw = (
    -1000.0
    + 1100.0
    * torch.exp(
        -4.0 * squared_radius_dhw
    )
)  # [D=16, H=32, W=20]


# PyTorch 3D interpolation contract에 맞게
# Batch와 Channel axis 추가
#
# [D, H, W]
#      ↓ unsqueeze(0)
# [C=1, D, H, W]
#      ↓ unsqueeze(0)
# [B=1, C=1, D, H, W]
synthetic_ct_bcdhw = (
    synthetic_ct_dhw
    .unsqueeze(dim=0)
    .unsqueeze(dim=0)
)  # [1, 1, 16, 32, 20]


# Cell 1에서 계산한 target Shape로 CT resampling
resampled_ct_bcdhw = resample_ct_image(
    ct_image_bcdhw=synthetic_ct_bcdhw,
    target_shape_dhw=target_shape_dhw,
)  # [1, 1, 32, 32, 60 argues]


# Resampling 전후 intensity 범위와 평균 계산
original_intensity_min = (
    synthetic_ct_bcdhw.min().item()
)

original_intensity_max = (
    synthetic_ct_bcdhw.max().item()
)

original_intensity_mean = (
    synthetic_ct_bcdhw.mean().item()
)

resampled_intensity_min = (
    resampled_ct_bcdhw.min().item()
)

resampled_intensity_max = (
    resampled_ct_bcdhw.max().item()
)

resampled_intensity_mean = (
    resampled_ct_bcdhw.mean().item()
)


print(
    "Original CT shape:",
    synthetic_ct_bcdhw.shape,
)

print(
    "Resampled CT shape:",
    resampled_ct_bcdhw.shape,
)

print(
    "Original dtype:",
    synthetic_ct_bcdhw.dtype,
)

print(
    "Resampled dtype:",
    resampled_ct_bcdhw.dtype,
)

print(
    "Original intensity range:",
    (
        original_intensity_min,
        original_intensity_max,
    ),
)

print(
    "Resampled intensity range:",
    (
        resampled_intensity_min,
        resampled_intensity_max,
    ),
)

print(
    "Original intensity mean:",
    original_intensity_mean,
)

print(
    "Resampled intensity mean:",
    resampled_intensity_mean,
)

Original CT shape: torch.Size([1, 1, 16, 32, 20])
Resampled CT shape: torch.Size([1, 1, 32, 32, 60])
Original dtype: torch.float32
Resampled dtype: torch.float32
Original intensity range: (-999.9932250976562, 64.2706298828125)
Resampled intensity range: (-999.9932250976562, 64.2706298828125)
Original intensity mean: -918.1396484375
Resampled intensity mean: -918.1396484375


In [25]:
# Cell 3 — Nearest-Neighbor로 Segmentation Label Resampling

def resample_segmentation_label(
    segmentation_target_bdhw: torch.Tensor,  # [B, D, H, W], torch.long
    target_shape_dhw: tuple[int, int, int],  # [D_target, H_target, W_target]
) -> torch.Tensor:                           # [B, D_target, H_target, W_target]
    """Hard segmentation label을 nearest-neighbor 방식으로 resampling."""
    
    # F.interpolate의 3D input contract에 맞게 Channel axis 추가:
    # [B, D, H, W] -> [B, C=1, D, H, W]
    target_bcdhw = segmentation_target_bdhw.unsqueeze(
        dim=1
    ).to(
        dtype=torch.float32
    ) # [B, 1, D, H, W]
    
    # 가장 가까운 원본 voxel의 class ID를 그대로 선택
    resampled_target_bcdhw = F.interpolate(
        target_bcdhw,
        size=target_shape_dhw,
        mode="nearest",
    )  # [B, 1, D_target, H_target, W_target]

    # 임시 Channel axis 제거 후 원래 정수 dtype 복원
    resampled_target_bdhw = (
        resampled_target_bcdhw
        .squeeze(dim=1)
        .to(dtype=segmentation_target_bdhw.dtype)
    )  # [B, D_target, H_target, W_target]

    return resampled_target_bdhw


# Class 1을 나타낼 첫 번째 가상 장기 영역 생성
#
# Volume 중심보다 D축 음의 방향에 배치한 ellipsoid
class_1_mask_dhw = (
    ((grid_d + 0.35) / 0.35).square()
    + (grid_h / 0.45).square()
    + (grid_w / 0.40).square()
    <= 1.0
)  # [D=16, H=32, W=20], torch.bool


# Class 2를 나타낼 두 번째 가상 장기 영역 생성
#
# Volume 중심보다 D축 양의 방향에 배치한 작은 ellipsoid
class_2_mask_dhw = (
    ((grid_d - 0.35) / 0.25).square()
    + ((grid_h - 0.20) / 0.30).square()
    + ((grid_w + 0.20) / 0.30).square()
    <= 1.0
)  # [D=16, H=32, W=20], torch.bool


# 모든 voxel이 background class 0인 label 생성
synthetic_label_dhw = torch.zeros(
    original_shape_dhw,
    dtype=torch.long,
)  # [D=16, H=32, W=20]


# 첫 번째 장기 영역에 class ID 1 할당
synthetic_label_dhw[
    class_1_mask_dhw
] = 1


# 두 번째 장기 영역에 class ID 2 할당
synthetic_label_dhw[
    class_2_mask_dhw
] = 2


# Segmentation target contract에 맞게 Batch axis 추가
#
# [D, H, W]
#      ↓
# [B=1, D, H, W]
synthetic_target_bdhw = synthetic_label_dhw.unsqueeze(
    dim=0
)  # [1, 16, 32, 20]


# 올바른 nearest-neighbor label resampling
resampled_target_bdhw = resample_segmentation_label(
    segmentation_target_bdhw=synthetic_target_bdhw,
    target_shape_dhw=target_shape_dhw,
)  # [1, 32, 32, 60]


# 비교를 위해 label에 잘못된 trilinear interpolation 적용
wrong_trilinear_target_bcdhw = F.interpolate(
    synthetic_target_bdhw
    .unsqueeze(dim=1)
    .to(dtype=torch.float32),
    size=target_shape_dhw,
    mode="trilinear",
    align_corners=False,
)  # [1, 1, 32, 32, 60]


# Trilinear 결과 중 정수가 아닌 voxel 식별
#
# 예:
# 0.25, 0.50, 0.75, 1.25 등
fractional_voxel_mask = (
    torch.abs(
        wrong_trilinear_target_bcdhw
        - torch.round(
            wrong_trilinear_target_bcdhw
        )
    )
    > 1e-6
)  # [1, 1, 32, 32, 60]


# 존재하지 않는 fractional class 값을 가진 voxel 수 계산
fractional_voxel_count = fractional_voxel_mask.sum().item()


# Resampling 전후의 class ID와 voxel 수 계산
(
    original_class_ids,
    original_class_counts,
) = torch.unique(
    synthetic_target_bdhw,
    return_counts=True,
)

(
    resampled_class_ids,
    resampled_class_counts,
) = torch.unique(
    resampled_target_bdhw,
    return_counts=True,
)


print(
    "Original target shape:",
    synthetic_target_bdhw.shape,
)

print(
    "Resampled target shape:",
    resampled_target_bdhw.shape,
)

print(
    "Original dtype:",
    synthetic_target_bdhw.dtype,
)

print(
    "Resampled dtype:",
    resampled_target_bdhw.dtype,
)

print(
    "Original class IDs:",
    original_class_ids,
)

print(
    "Resampled class IDs:",
    resampled_class_ids,
)

print(
    "Original class counts:",
    original_class_counts,
)

print(
    "Resampled class counts:",
    resampled_class_counts,
)

print(
    "Fractional voxels from wrong trilinear:",
    fractional_voxel_count,
)

Original target shape: torch.Size([1, 16, 32, 20])
Resampled target shape: torch.Size([1, 32, 32, 60])
Original dtype: torch.int64
Resampled dtype: torch.int64
Original class IDs: tensor([0, 1, 2])
Resampled class IDs: tensor([0, 1, 2])
Original class counts: tensor([9843,  296,  101])
Resampled class counts: tensor([59058,  1776,   606])
Fractional voxels from wrong trilinear: 2106


In [26]:
# Cell 4 — Resampling Round-Trip과 정보 손실 검증

def calculate_image_round_trip_error(
    original_image_bcdhw: torch.Tensor,  # [B, C, D, H, W]
    restored_image_bcdhw: torch.Tensor,  # [B, C, D, H, W]
) -> tuple[float, float]:
    """원본과 복원 image 사이의 MAE와 maximum error 계산."""

    # Voxel별 absolute intensity error 계산
    absolute_error_bcdhw = torch.abs(
        original_image_bcdhw
        - restored_image_bcdhw
    )  # [B, C, D, H, W]

    # 전체 voxel의 평균 absolute error 계산
    mean_absolute_error = (
        absolute_error_bcdhw.mean().item()
    )

    # 가장 큰 voxel intensity error 계산
    maximum_absolute_error = (
        absolute_error_bcdhw.max().item()
    )

    return (
        mean_absolute_error,
        maximum_absolute_error,
    )


def calculate_label_agreement(
    original_target_bdhw: torch.Tensor,  # [B, D, H, W]
    restored_target_bdhw: torch.Tensor,  # [B, D, H, W]
) -> float:
    """원본과 복원 label에서 동일한 class를 가진 voxel 비율 계산."""

    # 같은 위치에서 class ID가 일치하는 voxel 식별
    matching_voxel_mask = (
        original_target_bdhw
        == restored_target_bdhw
    )  # [B, D, H, W], torch.bool

    # Boolean mask를 float로 변환한 뒤 평균 계산:
    # 모든 voxel이 같으면 1.0
    voxel_agreement = (
        matching_voxel_mask
        .to(dtype=torch.float32)
        .mean()
        .item()
    )

    return voxel_agreement


# Target Shape로 resampling한 "CT"를 원래 Shape로 복원
restored_ct_bcdhw = F.interpolate(
    resampled_ct_bcdhw,
    size=original_shape_dhw,
    mode="trilinear",
    align_corners=False,
)  # [1, 1, 16, 32, 20]


# Target Shape로 resampling한 "label"을 원래 Shape로 복원
restored_target_bdhw = resample_segmentation_label(
    segmentation_target_bdhw=resampled_target_bdhw,
    target_shape_dhw=original_shape_dhw,
)  # [1, 16, 32, 20]


# CT image의 round-trip intensity error 계산
(
    ct_mean_absolute_error,
    ct_maximum_absolute_error,
) = calculate_image_round_trip_error(
    original_image_bcdhw=synthetic_ct_bcdhw,
    restored_image_bcdhw=restored_ct_bcdhw,
)


# Segmentation label의 round-trip voxel agreement 계산
label_voxel_agreement = calculate_label_agreement(
    original_target_bdhw=synthetic_target_bdhw,
    restored_target_bdhw=restored_target_bdhw,
)


# Label에서 원본과 달라진 voxel 개수 계산
changed_label_voxel_count = (
    synthetic_target_bdhw
    != restored_target_bdhw
).sum().item()


print("Standard round-trip")
print(
    "Original CT shape:",
    synthetic_ct_bcdhw.shape,
)

print(
    "Restored CT shape:",
    restored_ct_bcdhw.shape,
)

print(
    "CT mean absolute error:",
    ct_mean_absolute_error,
)

print(
    "CT maximum absolute error:",
    ct_maximum_absolute_error,
)

print(
    "Label voxel agreement:",
    label_voxel_agreement,
)

print(
    "Changed label voxels:",
    changed_label_voxel_count,
)


# ------------------------------------------------------------
# 작은 구조 소실을 확인하기 위한 downsampling stress test
# ------------------------------------------------------------

# 기존 segmentation target 복사
thin_structure_target_bdhw = (
    synthetic_target_bdhw.clone()
)  # [1, 16, 32, 20]


# 홀수 index 위치의 단일 voxel에 class 3 할당
#
# Downsampling 시 선택되는 grid에서 벗어나도록
# D, H, W 모두 홀수 index 사용
thin_structure_target_bdhw[
    0,
    7,
    15,
    9,
] = 3


# 각 spatial axis를 절반으로 줄인 coarse Shape 정의
coarse_shape_dhw: tuple[int, int, int] = (
    original_shape_dhw[0] // 2,
    original_shape_dhw[1] // 2,
    original_shape_dhw[2] // 2,
)


# 작은 구조가 포함된 label을 coarse grid로 downsampling
coarse_target_bdhw = resample_segmentation_label(
    segmentation_target_bdhw=(
        thin_structure_target_bdhw
    ),
    target_shape_dhw=coarse_shape_dhw,
)  # [1, 8, 16, 10]


# Coarse label을 다시 원본 Shape로 upsampling
restored_thin_target_bdhw = (
    resample_segmentation_label(
        segmentation_target_bdhw=(
            coarse_target_bdhw
        ),
        target_shape_dhw=original_shape_dhw,
    )
)  # [1, 16, 32, 20]


# 원본, coarse와 복원 label의 class ID 확인
original_thin_class_ids = torch.unique(
    thin_structure_target_bdhw
)

coarse_class_ids = torch.unique(
    coarse_target_bdhw
)

restored_thin_class_ids = torch.unique(
    restored_thin_target_bdhw
)


# Class 3 voxel 수 변화 측정
original_class_3_voxel_count = (
    thin_structure_target_bdhw == 3
).sum().item()

coarse_class_3_voxel_count = (
    coarse_target_bdhw == 3
).sum().item()

restored_class_3_voxel_count = (
    restored_thin_target_bdhw == 3
).sum().item()


print()
print("Thin-structure stress test")

print(
    "Original thin-label shape:",
    thin_structure_target_bdhw.shape,
)

print(
    "Coarse label shape:",
    coarse_target_bdhw.shape,
)

print(
    "Original class IDs:",
    original_thin_class_ids,
)

print(
    "Coarse class IDs:",
    coarse_class_ids,
)

print(
    "Restored class IDs:",
    restored_thin_class_ids,
)

print(
    "Original class-3 voxels:",
    original_class_3_voxel_count,
)

print(
    "Coarse class-3 voxels:",
    coarse_class_3_voxel_count,
)

print(
    "Restored class-3 voxels:",
    restored_class_3_voxel_count,
)

Standard round-trip
Original CT shape: torch.Size([1, 1, 16, 32, 20])
Restored CT shape: torch.Size([1, 1, 16, 32, 20])
CT mean absolute error: 1.3746484518051147
CT maximum absolute error: 17.636489868164062
Label voxel agreement: 1.0
Changed label voxels: 0

Thin-structure stress test
Original thin-label shape: torch.Size([1, 16, 32, 20])
Coarse label shape: torch.Size([1, 8, 16, 10])
Original class IDs: tensor([0, 1, 2, 3])
Coarse class IDs: tensor([0, 1, 2])
Restored class IDs: tensor([0, 1, 2])
Original class-3 voxels: 1
Coarse class-3 voxels: 0
Restored class-3 voxels: 0
